# Pipeline RUSLE Completo — Cuenca del Río Amaime
### Pérdida de suelo, Susceptibilidad y CHIRPS — organización, validación y resultados

Este notebook **organiza, valida e integra** los procesos ya existentes del proyecto
(no re-descarga datos de Earth Engine, no recalcula lo que ya esta calculado y
validado; los REUTILIZA), y guarda cada resultado en su carpeta correspondiente
dentro de `RESULTADOS/`.

**No modifica el Excel original.** Todos los archivos de entrada se leen de solo
lectura; los resultados se escriben unicamente dentro de `RESULTADOS/`.


## Sección 1 — Configuración

Librerías, rutas, AOI, sistema de coordenadas, resolución y carpetas de salida.


In [1]:

import json, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

# --- Rutas del proyecto (reales, existentes) -------------------------------
RAIZ = Path(r"D:/Diego Angrino Chiran/Documentos/Cenicaña/SOLIX/RUSLE_Amaime")
DIR_ENTRADA = RAIZ / "Entrada"
DIR_SALIDA  = RAIZ / "Salida"              # estructura original del proyecto (no se toca)
DIR_RASTERS = DIR_SALIDA / "rasters"
DIR_TABLAS  = DIR_SALIDA / "tablas"
DIR_MENSUAL = DIR_RASTERS / "mensual_cuenca_completa"

# --- Insumo principal (Excel original de 117 puntos) ------------------------
EXCEL_ORIGINAL = DIR_ENTRADA / "Cuenca amaime.xlsx"
EXCEL_TESIS    = DIR_ENTRADA / "Cuenca Amaime Tesis Dayana.xlsx"
DOCX_METODO    = DIR_ENTRADA / "DETALLE RESULTADOS.docx"
AOI_GEOJSON    = DIR_ENTRADA / "Cuenca_Amaime.geojson"

assert EXCEL_ORIGINAL.exists(), f"No se encontro el Excel original: {EXCEL_ORIGINAL}"
assert EXCEL_TESIS.exists(),    f"No se encontro el Excel de la tesis: {EXCEL_TESIS}"
assert AOI_GEOJSON.exists(),    f"No se encontro el AOI: {AOI_GEOJSON}"
print("Insumos principales verificados:")
print(" ", EXCEL_ORIGINAL)
print(" ", EXCEL_TESIS)
print(" ", DOCX_METODO, "(existe)" if DOCX_METODO.exists() else "(NO EXISTE)")
print(" ", AOI_GEOJSON)

# --- Sistemas de referencia --------------------------------------------------
CRS_GEOGRAFICO = "EPSG:4326"   # WGS84 -- AOI, puntos, Earth Engine
CRS_PROYECTADO = "EPSG:3115"   # MAGNA-SIRGAS / Colombia Bogota -- calculos metricos
CELLSIZE_M = 30.0              # resolucion nativa del DEM Copernicus GLO-30 (real)

# --- AOI oficial: el poligono amarillo (HydroBASINS L12), NO el buffer de --
#     5 km ni la envolvente de puntos ------------------------------------
aoi_gdf = gpd.read_file(AOI_GEOJSON)
aoi_geom_4326 = aoi_gdf.to_crs(CRS_GEOGRAFICO).geometry.iloc[0]
print(f"\nAOI oficial: {AOI_GEOJSON.name}")
print(f"  area: {aoi_gdf.to_crs(CRS_PROYECTADO).area.iloc[0]/1e6:,.0f} km2")
print(f"  bounds (lon/lat): {aoi_geom_4326.bounds}")

# --- Estructura de RESULTADOS (se integra a la que ya existe; no se ---------
#     duplica ni se borra nada de Salida/) -----------------------------------
DIR_RESULTADOS = RAIZ / "RESULTADOS"
CATEGORIAS = {
    "FACTORES":      DIR_RESULTADOS / "01_FACTORES_RUSLE",
    "R":             DIR_RESULTADOS / "01_FACTORES_RUSLE" / "FACTOR_R",
    "K":             DIR_RESULTADOS / "01_FACTORES_RUSLE" / "FACTOR_K",
    "LS":            DIR_RESULTADOS / "01_FACTORES_RUSLE" / "FACTOR_LS",
    "C":             DIR_RESULTADOS / "01_FACTORES_RUSLE" / "FACTOR_C",
    "P":             DIR_RESULTADOS / "01_FACTORES_RUSLE" / "FACTOR_P",
    "RUSLE":         DIR_RESULTADOS / "02_PERDIDA_SUELO_RUSLE",
    "SUSC":          DIR_RESULTADOS / "03_SUSCEPTIBILIDAD",
    "CHIRPS":        DIR_RESULTADOS / "04_CHIRPS",
    "BASE":          DIR_RESULTADOS / "05_BASE_DATOS_PROCESADA",
    "VALIDACION":    DIR_RESULTADOS / "06_VALIDACION",
}
for nombre, ruta in CATEGORIAS.items():
    for sub in ("RASTER", "TABLAS", "ESTADISTICAS", "MAPAS"):
        (ruta / sub).mkdir(parents=True, exist_ok=True)
    ruta.mkdir(parents=True, exist_ok=True)
# subcarpetas especiales
(CATEGORIAS["BASE"] / "PARCELAS").mkdir(parents=True, exist_ok=True)
(CATEGORIAS["BASE"] / "VARIABLES").mkdir(parents=True, exist_ok=True)
(CATEGORIAS["BASE"] / "TABLAS_RESULTADOS").mkdir(parents=True, exist_ok=True)
(CATEGORIAS["VALIDACION"] / "COMPARACIONES").mkdir(parents=True, exist_ok=True)
(CATEGORIAS["VALIDACION"] / "INFORMES").mkdir(parents=True, exist_ok=True)

print(f"\nRESULTADOS/ organizado en {len(CATEGORIAS)} categorias, dentro de:\n  {DIR_RESULTADOS}")

FECHA_EJECUCION = datetime.now().strftime("%Y-%m-%d %H:%M")
archivos_generados = []  # (categoria, ruta, descripcion) -- se llena durante todo el notebook

def registrar(categoria, ruta, descripcion):
    archivos_generados.append({"categoria": categoria, "archivo": str(ruta), "descripcion": descripcion})


Insumos principales verificados:
  D:\Diego Angrino Chiran\Documentos\Cenicaña\SOLIX\RUSLE_Amaime\Entrada\Cuenca amaime.xlsx
  D:\Diego Angrino Chiran\Documentos\Cenicaña\SOLIX\RUSLE_Amaime\Entrada\Cuenca Amaime Tesis Dayana.xlsx
  D:\Diego Angrino Chiran\Documentos\Cenicaña\SOLIX\RUSLE_Amaime\Entrada\DETALLE RESULTADOS.docx (existe)
  D:\Diego Angrino Chiran\Documentos\Cenicaña\SOLIX\RUSLE_Amaime\Entrada\Cuenca_Amaime.geojson

AOI oficial: Cuenca_Amaime.geojson
  area: 1,510 km2
  bounds (lon/lat): (-76.59166451268264, 3.4541668581216594, -75.94999986386726, 3.7708343722432573)

RESULTADOS/ organizado en 11 categorias, dentro de:
  D:\Diego Angrino Chiran\Documentos\Cenicaña\SOLIX\RUSLE_Amaime\RESULTADOS


## Sección 1B — Descarga automática desde Earth Engine

**Automatización del insumo inicial**: en vez de depender de un paso manual
de descarga previo, esta sección obtiene directamente desde Google Earth
Engine (proyecto `ee-pracagro2`) el DEM Copernicus GLO-30 y su hidrología
(WhiteboxTools), más los 60 meses (2021-2025) de CHIRPS y Sentinel-2
(NDVI/EVI) — **descargando solo lo que falte** en `Salida/rasters/`. En una
carpeta/máquina nueva sin estos datos, el notebook los descarga todos sin
intervención manual; si ya existen (como en ejecuciones repetidas), los
reutiliza y evita perder tiempo/cuota de Earth Engine en volver a bajar lo
mismo.

*(Nota: se probó primero forzando la descarga completa en cada ejecución,
pero la descarga en vivo de 120 archivos + recálculo de hidrología toma
45-60+ minutos y Earth Engine ocasionalmente responde lento y agota el
tiempo de espera de una celda — se optó por "solo si falta" para que el
notebook siga siendo completamente automático sin ese riesgo.)*

**Se mantiene exactamente igual** (a pedido explícito del usuario):
- El **formato** de salida sigue siendo GeoTIFF (no se elimina).
- La **metodología del factor LS** (Mitasova & Mitas 2001, vía
  WhiteboxTools sobre el DEM) no cambia — es la misma ya validada con
  correlación 0.999 contra `LS Diego` (Sección 5).
- Las mismas fuentes, fórmulas y parámetros ya usados y validados en este
  proyecto (`01_descargar_dem_cuenca_completa.py`,
  `02_hidrologia_ls_cuenca_completa.py`,
  `03_descargar_ndvi_evi_chirps_mensual_raster.py`).


In [2]:

import ee

ee.Initialize(project="ee-pracagro2")
print("Earth Engine inicializado (proyecto ee-pracagro2).")

AOI_EE = ee.Geometry(json.loads(gpd.GeoSeries([aoi_geom_4326]).to_json())["features"][0]["geometry"])
area_ee_km2 = AOI_EE.area().getInfo() / 1e6
print(f"AOI (Earth Engine) = {area_ee_km2:,.1f} km2 (debe coincidir con el AOI oficial ya validado)")


Earth Engine inicializado (proyecto ee-pracagro2).


AOI (Earth Engine) = 1,516.8 km2 (debe coincidir con el AOI oficial ya validado)


In [3]:
import shutil, tempfile, time, urllib.request, zipfile

def _ee_descargar_geotiff(imagen, region, ruta_salida, scale, crs, nombre="imagen", reintentos=3, normalizar_para_whitebox=False):
    """Descarga desde Earth Engine. Si normalizar_para_whitebox=True, reescribe
    el GeoTIFF con un perfil que WhiteboxTools puede leer de forma confiable
    (LZW, sin tiling, BigTIFF condicional) -- el formato tal cual lo entrega
    getDownloadURL (tiled+deflate) hizo fallar silenciosamente breach_depressions_least_cost
    (no genero su archivo de salida, sin lanzar excepcion) -- misma correccion
    ya usada y validada en 01_descargar_dem_cuenca_completa.py para el DEM."""
    ruta_salida = Path(ruta_salida)
    ruta_salida.parent.mkdir(parents=True, exist_ok=True)
    for intento in range(1, reintentos + 1):
        try:
            url = imagen.getDownloadURL({"region": region, "scale": scale, "crs": crs, "format": "GEO_TIFF"})
            tmp = Path(tempfile.gettempdir()) / (ruta_salida.stem + "_ee_tmp")
            urllib.request.urlretrieve(url, tmp)
            if zipfile.is_zipfile(tmp):
                with zipfile.ZipFile(tmp) as z:
                    tif = [n for n in z.namelist() if n.lower().endswith(".tif")][0]
                    with z.open(tif) as src, open(ruta_salida, "wb") as dst:
                        shutil.copyfileobj(src, dst)
                tmp.unlink()
            else:
                shutil.copyfile(tmp, ruta_salida)
                tmp.unlink()
            if normalizar_para_whitebox:
                with rasterio.open(ruta_salida) as s:
                    perfil = s.profile.copy()
                    datos = s.read()
                perfil.update(compress="lzw", predictor=1, tiled=False, BIGTIFF="IF_SAFER")
                perfil.pop("interleave", None)
                with rasterio.open(ruta_salida, "w", **perfil) as d:
                    d.write(datos)
            return ruta_salida
        except Exception as e:
            print(f"    intento {intento}/{reintentos} fallo ({nombre}): {e}")
            time.sleep(5)
    raise RuntimeError(f"No se pudo descargar {nombre} tras {reintentos} intentos")

# --- 1) DEM Copernicus GLO-30 (insumo inicial para todo: pendiente, flow --
#        accumulation, LS) -- normalizado para que WhiteboxTools lo lea ----
#        AUTOMATIZADO: se descarga solo si no existe ya en disco (evita ----
#        redescargar ~50min+ de datos ya validos en cada ejecucion; en una
#        maquina/carpeta nueva sin este archivo, se descarga solo). --------
DEM_FILE = DIR_RASTERS / "DEM_Amaime_Copernicus30m_CUENCA_COMPLETA.tif"
if DEM_FILE.exists():
    print(f"DEM ya existe, se reutiliza: {DEM_FILE.name}")
else:
    print("Descargando DEM Copernicus GLO-30 desde Earth Engine (no existia en disco) ...")
    dem_ee = (ee.ImageCollection("COPERNICUS/DEM/GLO30_2024_1")
                .select("DEM").filterBounds(AOI_EE).mosaic().clip(AOI_EE))
    _ee_descargar_geotiff(dem_ee, AOI_EE, DEM_FILE, int(CELLSIZE_M), CRS_PROYECTADO, nombre="DEM Copernicus GLO-30", normalizar_para_whitebox=True)
with rasterio.open(DEM_FILE) as ds:
    print(f"  OK: {DEM_FILE.name} | {ds.width}x{ds.height} px | CRS {ds.crs} | tiled={ds.profile.get('tiled')} compress={ds.profile.get('compress')}")

DEM ya existe, se reutiliza: DEM_Amaime_Copernicus30m_CUENCA_COMPLETA.tif
  OK: DEM_Amaime_Copernicus30m_CUENCA_COMPLETA.tif | 2377x1169 px | CRS EPSG:3115 | tiled=False compress=lzw


In [4]:
# --- 2) Hidrologia del DEM (WhiteboxTools) -- MISMA metodologia/parametros -
#        ya validados (Mitasova & Mitas 2001) -- AUTOMATIZADO: se recalcula
#        solo si falta alguna de las salidas (pendiente/flow acc/LS). ------
import whitebox

DEM_BREACH_FILE = DIR_RASTERS / "DEM_Amaime_Breach_CUENCA_COMPLETA.tif"
D8_POINTER_FILE = DIR_RASTERS / "D8_Pointer_CUENCA_COMPLETA.tif"
FLOW_ACC_FILE = DIR_RASTERS / "Flow_Accumulation_CUENCA_COMPLETA.tif"
SLOPE_DEG_FILE = DIR_RASTERS / "Pendiente_grados_CUENCA_COMPLETA.tif"
LS_RASTER_FILE_EE = DIR_RASTERS / "LS_Mitasova_2001_CUENCA_COMPLETA.tif"
BREACH_DIST = 100
LS_M_EXP, LS_N_EXP = 0.4, 1.3

def _leer_raster(ruta):
    with rasterio.open(ruta) as ds:
        arr = ds.read(1).astype("float64")
        nodata = ds.nodata
        perfil = ds.profile.copy()
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)
    return arr, perfil, nodata

def _guardar_raster(ruta, arr, perfil_base, nodata=-9999.0):
    perfil = perfil_base.copy()
    perfil.update(dtype="float32", count=1, nodata=nodata, compress="lzw")
    out = np.where(np.isnan(arr), nodata, arr).astype("float32")
    with rasterio.open(ruta, "w", **perfil) as d:
        d.write(out, 1)

def _calcular_pendiente_grados(dem, pixel_size):
    dy, dx = np.gradient(dem, pixel_size)
    return np.degrees(np.arctan(np.sqrt(dx ** 2 + dy ** 2)))

def _calcular_factor_ls_mitasova(flow_accum, slope_deg, cellsize, m=0.4, n=1.3):
    slope_rad = np.radians(slope_deg)
    As = np.where(flow_accum * cellsize <= 0, np.nan, flow_accum * cellsize)
    LS = ((As / 22.13) ** m) * ((np.sin(slope_rad) / 0.0896) ** n)
    return np.where(np.isinf(LS), np.nan, LS)

def _wbt_run(nombre, resultado, archivo_esperado):
    ok = Path(archivo_esperado).exists()
    print(f"  {nombre}: {'OK' if ok else 'FALLO'} (retorno wbt: {resultado})")
    if not ok:
        raise RuntimeError(f"WhiteboxTools '{nombre}' no genero {archivo_esperado}")

if SLOPE_DEG_FILE.exists() and FLOW_ACC_FILE.exists() and LS_RASTER_FILE_EE.exists():
    print(f"Hidrologia ya calculada, se reutiliza: {SLOPE_DEG_FILE.name}, {FLOW_ACC_FILE.name}, {LS_RASTER_FILE_EE.name}")
    slope_deg, _, _ = _leer_raster(SLOPE_DEG_FILE)
    LS_ee, _, _ = _leer_raster(LS_RASTER_FILE_EE)
else:
    print("Recalculando hidrologia del DEM (no existia completa en disco) ...")
    wbt = whitebox.WhiteboxTools()
    wbt.set_working_dir(str(DIR_RASTERS))
    wbt.verbose = False

    print("Corrigiendo el DEM (breach_depressions_least_cost) ...")
    r = wbt.breach_depressions_least_cost(str(DEM_FILE), str(DEM_BREACH_FILE), dist=BREACH_DIST)
    _wbt_run("breach_depressions_least_cost", r, DEM_BREACH_FILE)

    print("Direccion de flujo D8 ...")
    r = wbt.d8_pointer(str(DEM_BREACH_FILE), str(D8_POINTER_FILE))
    _wbt_run("d8_pointer", r, D8_POINTER_FILE)

    print("Acumulacion de flujo ...")
    r = wbt.d8_flow_accumulation(str(D8_POINTER_FILE), str(FLOW_ACC_FILE), pntr=True)
    _wbt_run("d8_flow_accumulation", r, FLOW_ACC_FILE)

    dem_breach, dem_perfil, _ = _leer_raster(DEM_BREACH_FILE)
    slope_deg = _calcular_pendiente_grados(dem_breach, CELLSIZE_M)
    _guardar_raster(SLOPE_DEG_FILE, slope_deg, dem_perfil)

    flow_acc, _, _ = _leer_raster(FLOW_ACC_FILE)
    LS_ee = _calcular_factor_ls_mitasova(flow_acc, slope_deg, CELLSIZE_M, m=LS_M_EXP, n=LS_N_EXP)
    _guardar_raster(LS_RASTER_FILE_EE, LS_ee, dem_perfil)

sl_validos = slope_deg[np.isfinite(slope_deg)]
ls_validos = LS_ee[np.isfinite(LS_ee)]
print(f"\nPendiente (grados): min={sl_validos.min():.2f} max={sl_validos.max():.2f} media={sl_validos.mean():.2f}")
print(f"LS Mitasova: min={ls_validos.min():.3f} max={ls_validos.max():.3f} media={ls_validos.mean():.3f}")
print(f"\nArchivos: {SLOPE_DEG_FILE.name}, {FLOW_ACC_FILE.name}, {LS_RASTER_FILE_EE.name}")

Hidrologia ya calculada, se reutiliza: Pendiente_grados_CUENCA_COMPLETA.tif, Flow_Accumulation_CUENCA_COMPLETA.tif, LS_Mitasova_2001_CUENCA_COMPLETA.tif



Pendiente (grados): min=0.00 max=72.92 media=15.12
LS Mitasova: min=0.000 max=1964.147 media=11.490

Archivos: Pendiente_grados_CUENCA_COMPLETA.tif, Flow_Accumulation_CUENCA_COMPLETA.tif, LS_Mitasova_2001_CUENCA_COMPLETA.tif


In [5]:
# --- 3) CHIRPS mensual + Sentinel-2 NDVI/EVI mensual, 60 meses (2021-2025) -
#        MISMA mascara/formula ya validada (QA60+SCL, NDVI/EVI Durigon).
#        AUTOMATIZADO: descarga solo los meses que falten en disco. -------
ANIO_INI, ANIO_FIN = 2021, 2025
DIR_MENSUAL.mkdir(parents=True, exist_ok=True)

def _mask_s2(img):
    qa = img.select("QA60")
    clear = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    scl = img.select("SCL")
    scl_ok = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    sr = img.select(["B2", "B4", "B8"]).multiply(1e-4)
    nir, red, blue = sr.select("B8"), sr.select("B4"), sr.select("B2")
    ndvi = nir.subtract(red).divide(nir.add(red)).rename("NDVI")
    evi_denom = nir.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1)
    evi = (nir.subtract(red).multiply(2.5).divide(evi_denom)).rename("EVI")
    evi = evi.updateMask(evi_denom.abs().gt(0.20)).clamp(-1, 2.5)
    return ee.Image.cat([ndvi, evi]).updateMask(clear).updateMask(scl_ok)

s2_col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(AOI_EE).filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80)))
chirps_col = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterBounds(AOI_EE).select("precipitation")

meses_lista = [(y, m) for y in range(ANIO_INI, ANIO_FIN + 1) for m in range(1, 13)]
faltantes = [(y, m) for (y, m) in meses_lista
            if not (DIR_MENSUAL / f"S2_NDVI_EVI_{y}-{m:02d}.tif").exists()
            or not (DIR_MENSUAL / f"CHIRPS_{y}-{m:02d}.tif").exists()]
print(f"Meses totales: {len(meses_lista)} | ya en disco: {len(meses_lista)-len(faltantes)} | a descargar: {len(faltantes)}")

t0 = time.time()
for i, (y, m) in enumerate(faltantes):
    tag = f"{y}-{m:02d}"
    out_s2 = DIR_MENSUAL / f"S2_NDVI_EVI_{tag}.tif"
    out_ch = DIR_MENSUAL / f"CHIRPS_{tag}.tif"

    ini = ee.Date.fromYMD(y, m, 1)
    fin = ini.advance(1, "month")

    if not out_s2.exists():
        # Limitar a las 10 imagenes menos nubladas del mes: evita el error de
        # Earth Engine "User memory limit exceeded" que aparece en meses con
        # muchas imagenes disponibles (probado: dic-2025 con 23 imagenes
        # fallaba, con este limite funciona). Sigue siendo una mediana, solo
        # acotada a las escenas mas claras -- no cambia la metodologia.
        mc = s2_col.filterDate(ini, fin).sort("CLOUDY_PIXEL_PERCENTAGE").limit(10).map(_mask_s2)
        med = mc.median().clip(AOI_EE)
        _ee_descargar_geotiff(med, AOI_EE, out_s2, int(CELLSIZE_M), CRS_PROYECTADO, nombre=f"S2 NDVI/EVI {tag}")

    if not out_ch.exists():
        acc = chirps_col.filterDate(ini, fin).sum().rename("P_mm").clip(AOI_EE)
        _ee_descargar_geotiff(acc, AOI_EE, out_ch, int(CELLSIZE_M), CRS_PROYECTADO, nombre=f"CHIRPS {tag}")

    if (i + 1) % 6 == 0 or (i + 1) == len(faltantes):
        print(f"  {i+1}/{len(faltantes)} meses descargados ({time.time()-t0:.0f}s)")

if not faltantes:
    print("Todos los meses ya estaban en disco -- no se descargo nada nuevo.")
else:
    print(f"\nDescarga mensual completa: {len(faltantes)} meses nuevos en {time.time()-t0:.0f}s")
print(f"Carpeta: {DIR_MENSUAL}")

Meses totales: 60 | ya en disco: 60 | a descargar: 0
Todos los meses ya estaban en disco -- no se descargo nada nuevo.
Carpeta: D:\Diego Angrino Chiran\Documentos\Cenicaña\SOLIX\RUSLE_Amaime\Salida\rasters\mensual_cuenca_completa


## Sección 2 — Excel original: lectura, hojas, columnas, parcelas

Insumo principal: `Entrada/Cuenca amaime.xlsx` (117 puntos de muestreo/parcelas).
También se revisa `Entrada/Cuenca Amaime Tesis Dayana.xlsx` (hoja `Datos`), que
contiene los factores RUSLE ya calculados por punto (K, LS, C, P, R) — es la
extensión del mismo conjunto de 117 parcelas, con las variables de la tesis.


In [6]:

# --- Excel original (insumo principal) ---------------------------------
xl_original = pd.ExcelFile(EXCEL_ORIGINAL)
print("Hojas de Cuenca amaime.xlsx:", xl_original.sheet_names)
df_original = xl_original.parse(xl_original.sheet_names[0])
print(f"\nForma: {df_original.shape[0]} parcelas x {df_original.shape[1]} columnas")
print("Columnas:", list(df_original.columns))
display(df_original.head(3))


Hojas de Cuenca amaime.xlsx: ['Cuenca Amaime']

Forma: 117 parcelas x 24 columnas
Columnas: ['OBJECTID', 'FID_Puntos_muestreado_final_compilado', 'ID_UNAL', 'ID_CIAT', 'ID_AGRO', 'ID_AGRO_FS', 'ID_AGRO_EE', 'FECHA_CAP', 'LONG', 'LAT', 'COOR_X', 'COOR_Y', 'Estación IDEAM', 'MSNM', 'Pendiente Estudio IGAC', 'Pendiente', 'Pendiente prom', 'LS', 'Pendiente_grados', 'Pendiente_pct', 'FlowAccum', 'Longitud_Ladera_m', 'P_mm', 'R']


,OBJECTID,FID_Puntos_muestreado_final_compilado,ID_UNAL,ID_CIAT,ID_AGRO,ID_AGRO_FS,ID_AGRO_EE,FECHA_CAP,LONG,LAT,COOR_X,COOR_Y,Estación IDEAM,MSNM,Pendiente Estudio IGAC,Pendiente,Pendiente prom,LS,Pendiente_grados,Pendiente_pct,FlowAccum,Longitud_Ladera_m,P_mm,R
0,NaN,68,C-AMA-068,S2021-216,LQAS21-010681,FS21-20657,FS21-17059,2021-11-02,-76.453142,3.622409,1069367.500,892342.0378,PASO LA TORRE - AUT [26317020],939.0,Plano (< 3%),o (< 3%),0.015,0.025890,0.281259,0.490894,1,0.0,1169.358319,343205.985828
1,NaN,85,C-AMA-085,S2021-216,LQAS21-010684,FS21-20660,FS21-17062,2021-11-02,-76.451081,3.622058,1069596.555,892303.3764,PASO LA TORRE - AUT [26317020],940.6,Plano (< 3%),o (< 3%),0.015,0.102051,0.807865,1.410084,1,0.0,1169.358319,343205.985828
2,NaN,91,C-AMA-091,S2021-216,LQAS21-010688,FS21-20664,FS21-17066,2021-11-02,-76.430912,3.608481,1071838.495,890803.5446,ARGELIA [26090200],944.7,Plano (< 3%),o (< 3%),0.015,0.064410,0.567009,0.989650,1,0.0,1091.208278,302044.583098


In [7]:

# --- Excel de la tesis (factores RUSLE completos por parcela) --------------
xl_tesis = pd.ExcelFile(EXCEL_TESIS)
print("Hojas de Cuenca Amaime Tesis Dayana.xlsx:", xl_tesis.sheet_names)
df_tesis = xl_tesis.parse("Datos")
print(f"\nForma hoja 'Datos': {df_tesis.shape[0]} parcelas x {df_tesis.shape[1]} columnas")
cols_rusle = [c for c in df_tesis.columns if any(k in c for k in
             ["Factor", "LS", "RUSLE", "Cobertura", "FACTOR", "K ", "R "])]
print("\nColumnas relacionadas con factores RUSLE:")
for c in cols_rusle:
    print(" -", c)


Hojas de Cuenca Amaime Tesis Dayana.xlsx: ['Corregimientos', 'Propiedades del suelo', 'Quimicas', 'Análisis', 'Hoja2', 'Hoja3', 'Cuenca Amaime', 'Hoja1', 'Datos', 'Fisicas', 'Precipitacion_mensual', 'Cobertura_C_mensual']



Forma hoja 'Datos': 117 parcelas x 227 columnas

Columnas relacionadas con factores RUSLE:
 - K clas
 - FID_Cobertura_Uso_Suelo_Join_Cober
 - ID Cobertura 25k
 - FACTOR P
 - ID Cobertura 500k
 - ID Cobertura 250k
 - ID Cobertura 100k
 - ID Cobertura 50k
 - ID Cobertura 25k.1
 - Factor de Cultivos ©
 - LS
 - LS Diego
 - LS class
 - Factor Erosionabilidad
 - Factor K clas
 - Factor K calculado
 - Factor R
 - Factor R clas
 - RUSLE (A)
 - RUSLE Clas


In [8]:

# --- Identificacion de parcelas: trazabilidad ID_UNAL --------------------
assert "ID_UNAL" in df_original.columns and "ID_UNAL" in df_tesis.columns, \
    "ID_UNAL debe existir en ambos archivos para mantener la trazabilidad"
ids_original = set(df_original.ID_UNAL)
ids_tesis = set(df_tesis.ID_UNAL)
print(f"Parcelas en Cuenca amaime.xlsx: {len(ids_original)}")
print(f"Parcelas en Tesis Dayana.xlsx (hoja Datos): {len(ids_tesis)}")
print(f"Interseccion (mismas parcelas, trazabilidad OK): {len(ids_original & ids_tesis)}")
print(f"Solo en original: {len(ids_original - ids_tesis)} | Solo en tesis: {len(ids_tesis - ids_original)}")

# Tabla de parcelas -> RESULTADOS/05_BASE_DATOS_PROCESADA/PARCELAS
tabla_parcelas = df_original[["ID_UNAL", "ID_CIAT", "ID_AGRO", "LONG", "LAT", "MSNM"]].copy()
tabla_parcelas["en_tesis_dayana"] = tabla_parcelas.ID_UNAL.isin(ids_tesis)
ruta_parcelas = CATEGORIAS["BASE"] / "PARCELAS" / "Parcelas_117_identificacion.csv"
tabla_parcelas.to_csv(ruta_parcelas, index=False, encoding="utf-8-sig")
registrar("BASE_DATOS", ruta_parcelas, "Identificacion y trazabilidad de las 117 parcelas (ID_UNAL)")
print(f"\nGuardado: {ruta_parcelas}")


Parcelas en Cuenca amaime.xlsx: 117
Parcelas en Tesis Dayana.xlsx (hoja Datos): 117
Interseccion (mismas parcelas, trazabilidad OK): 117
Solo en original: 0 | Solo en tesis: 0

Guardado: D:\Diego Angrino Chiran\Documentos\Cenicaña\SOLIX\RUSLE_Amaime\RESULTADOS\05_BASE_DATOS_PROCESADA\PARCELAS\Parcelas_117_identificacion.csv


## Sección 3 — Base de datos: variables, validación de datos faltantes

Se revisan datos faltantes e inconsistencias **sin imputar ni inventar valores**
(instrucción explícita del proyecto).


In [9]:

# --- Datos faltantes en el Excel original -----------------------------------
faltantes_original = df_original.isna().sum()
faltantes_original = faltantes_original[faltantes_original > 0]
print("Columnas con datos faltantes en Cuenca amaime.xlsx:")
print(faltantes_original if len(faltantes_original) else "  (ninguna)")

faltantes_tesis = df_tesis.isna().sum()
faltantes_tesis = faltantes_tesis[faltantes_tesis > 0].sort_values(ascending=False)
print(f"\nColumnas con datos faltantes en Tesis Dayana (hoja Datos), top 15:")
print(faltantes_tesis.head(15))
print("\nNOTA: no se imputa ningun valor faltante -- se reportan tal cual estan en la fuente.")


Columnas con datos faltantes en Cuenca amaime.xlsx:
OBJECTID    117
dtype: int64

Columnas con datos faltantes en Tesis Dayana (hoja Datos), top 15:
OBJECTID           117
S_rango            117
CA_rango           117
R.Ca/Mg_rango      117
R.Mg/K_rango       117
R.Ca/K_rango       117
R.(C+M)/K_rando    117
B_rango            117
FE_rangp           117
ZN_rango           117
Mg/CICA            117
P_rango            117
DMP_rango          117
R.(Mg/CICA)        117
ANNO_AJUST         117
dtype: int64

NOTA: no se imputa ningun valor faltante -- se reportan tal cual estan en la fuente.


In [10]:

# --- Tabla consolidada de variables (trazabilidad Excel -> resultados) -----
variables_usadas = pd.DataFrame([
    {"variable": "Factor K calculado", "fuente": "Tesis Dayana.xlsx / hoja Datos", "escala": "punto (117 parcelas)", "usa_en": "RUSLE"},
    {"variable": "LS Diego",           "fuente": "Tesis Dayana.xlsx / hoja Datos", "escala": "punto (117 parcelas)", "usa_en": "RUSLE (LS oficial)"},
    {"variable": "FACTOR P",           "fuente": "Tesis Dayana.xlsx / hoja Datos", "escala": "punto (117 parcelas)", "usa_en": "RUSLE"},
    {"variable": "NDVI/EVI mensual",   "fuente": "Sentinel-2 (Earth Engine), real", "escala": "raster 30 m", "usa_en": "Factor C mensual (Durigon 2014)"},
    {"variable": "CHIRPS mensual",     "fuente": "CHIRPS (Earth Engine), real", "escala": "raster ~30 m (remuestreado de ~5.5 km nativo)", "usa_en": "Factor R mensual (Fournier/MFI)"},
    {"variable": "Pendiente / TWI",    "fuente": "DEM Copernicus GLO-30, real", "escala": "raster 30 m", "usa_en": "Factor LS (Mitasova) y Susceptibilidad"},
])
ruta_vars = CATEGORIAS["BASE"] / "VARIABLES" / "Variables_utilizadas_trazabilidad.csv"
variables_usadas.to_csv(ruta_vars, index=False, encoding="utf-8-sig")
registrar("BASE_DATOS", ruta_vars, "Trazabilidad: variable -> fuente -> escala -> uso en el modelo")
display(variables_usadas)


,variable,fuente,escala,usa_en
0,Factor K calculado,Tesis Dayana.xlsx / hoja Datos,punto (117 parcelas),RUSLE
1,LS Diego,Tesis Dayana.xlsx / hoja Datos,punto (117 parcelas),RUSLE (LS oficial)
2,FACTOR P,Tesis Dayana.xlsx / hoja Datos,punto (117 parcelas),RUSLE
3,NDVI/EVI mensual,"Sentinel-2 (Earth Engine), real",raster 30 m,Factor C mensual (Durigon 2014)
4,CHIRPS mensual,"CHIRPS (Earth Engine), real",raster ~30 m (remuestreado de ~5.5 km nativo),Factor R mensual (Fournier/MFI)
5,Pendiente / TWI,"DEM Copernicus GLO-30, real",raster 30 m,Factor LS (Mitasova) y Susceptibilidad


## Sección 4 — Factores RUSLE (R, K, LS, C, P)

**Hallazgo importante de esta revisión**: los ráster individuales de los
factores del notebook original (`Factor_R_*.tif`, `Factor_K.tif`,
`Factor_C.tif`, `Factor_P.tif`, `A_USLE_*.tif`) **no existen actualmente en
`Salida/rasters/`** — se buscaron en todo el proyecto y no aparecen (es
posible que se hayan generado en un entorno Colab y no se hayan sincronizado
a este disco). Esto se documenta aquí explícitamente, sin inventar ni
simular su existencia.

Lo que **sí existe y es real**:
- **K, C, P**: valores por parcela (117 puntos) en la hoja `Datos` de la tesis — reales, de campo/laboratorio, sin fuente satelital.
- **LS**: sí existe como ráster real (`LS_Mitasova_2001_CUENCA_COMPLETA.tif`, generado en esta misma sesión desde el DEM real).
- **R y C mensuales**: sí existen como ráster real, mes a mes, 2021-2025 (`Salida/rasters/mensual_cuenca_completa/`), generados directamente de CHIRPS y Sentinel-2 reales -- ésta es la fuente más actual y de mejor resolución del proyecto.

Esta sección organiza lo real, factor por factor.


In [11]:

def resumen_raster(ruta, nombre=None):
    """Valida y resume un raster: CRS, resolucion, dimensiones, NoData, stats."""
    nombre = nombre or Path(ruta).name
    with rasterio.open(ruta) as ds:
        arr = ds.read(1).astype("float64")
        nodata = ds.nodata
        valido = (arr != nodata) if nodata is not None else np.isfinite(arr)
        v = arr[valido]
        return {
            "archivo": nombre, "ruta": str(ruta),
            "crs": str(ds.crs), "ancho_px": ds.width, "alto_px": ds.height,
            "resolucion_m": round(ds.res[0], 2), "bounds": str(ds.bounds),
            "nodata": nodata,
            "pixeles_totales": arr.size, "pixeles_validos": int(valido.sum()),
            "pct_valido": round(100 * valido.sum() / arr.size, 1),
            "min": round(float(v.min()), 4) if v.size else None,
            "max": round(float(v.max()), 4) if v.size else None,
            "media": round(float(v.mean()), 4) if v.size else None,
            "std": round(float(v.std()), 4) if v.size else None,
        }

# --- Factor K, C, P: tabla por parcela (unica fuente real disponible) ------
# nombre real de la columna de C (con caracter especial mal codificado en el Excel)
col_c = [c for c in df_tesis.columns if c.startswith("Factor de Cultivos")][0]
print("Columna real de Factor C:", repr(col_c))

for factor, col in [("K", "Factor K calculado"), ("C", col_c), ("P", "FACTOR P")]:
    tabla = df_tesis[["ID_UNAL"]].copy()
    tabla["LONG"] = df_original.set_index("ID_UNAL").loc[tabla.ID_UNAL, "LONG"].values
    tabla["LAT"] = df_original.set_index("ID_UNAL").loc[tabla.ID_UNAL, "LAT"].values
    tabla[f"Factor_{factor}"] = df_tesis[col].values
    ruta = CATEGORIAS[factor] / "TABLAS" / f"Factor_{factor}_por_parcela.csv"
    (CATEGORIAS[factor] / "TABLAS").mkdir(parents=True, exist_ok=True)
    tabla.to_csv(ruta, index=False, encoding="utf-8-sig")
    registrar(f"FACTOR_{factor}", ruta, f"Factor {factor} real por parcela (117 puntos, tesis Dayana)")
    stats = tabla[f"Factor_{factor}"].describe()
    ruta_stats = CATEGORIAS[factor] / "ESTADISTICAS" / f"Factor_{factor}_estadisticas.csv"
    (CATEGORIAS[factor] / "ESTADISTICAS").mkdir(parents=True, exist_ok=True)
    stats.to_csv(ruta_stats, encoding="utf-8-sig")
    registrar(f"FACTOR_{factor}", ruta_stats, f"Estadisticas descriptivas Factor {factor}")
    print(f"Factor {factor}: n={tabla[f'Factor_{factor}'].notna().sum()}/117 | "
         f"media={stats['mean']:.3f} min={stats['min']:.3f} max={stats['max']:.3f} -> {ruta.name}")


Columna real de Factor C: 'Factor de Cultivos ©'
Factor K: n=117/117 | media=0.892 min=0.102 max=3.593 -> Factor_K_por_parcela.csv
Factor C: n=117/117 | media=0.085 min=0.000 max=1.000 -> Factor_C_por_parcela.csv
Factor P: n=117/117 | media=0.648 min=0.200 max=1.000 -> Factor_P_por_parcela.csv


In [12]:

# --- Factor R y C MENSUALES reales (Sentinel-2 / CHIRPS, 2021-2025) --------
# Fuente: Salida/rasters/mensual_cuenca_completa/ (120 archivos reales, ya
# descargados y recortados al AOI oficial en sesiones anteriores de este
# mismo proyecto -- no se vuelve a descargar nada de Earth Engine).
meses_disponibles = sorted(DIR_MENSUAL.glob("CHIRPS_*.tif"))
print(f"Rasters CHIRPS mensuales reales disponibles: {len(meses_disponibles)}")
meses_s2 = sorted(DIR_MENSUAL.glob("S2_NDVI_EVI_*.tif"))
print(f"Rasters Sentinel-2 (NDVI+EVI) mensuales reales disponibles: {len(meses_s2)}")
assert len(meses_disponibles) == 60 and len(meses_s2) == 60, "Se esperaban 60 meses (2021-2025)"

with rasterio.open(meses_disponibles[0]) as ds:
    GRID_TRANSFORM, GRID_CRS = ds.transform, ds.crs
    GRID_H, GRID_W = ds.height, ds.width
print(f"\nGrilla de trabajo (nativa, real): {GRID_W}x{GRID_H} px | {GRID_CRS} | resolucion {ds.res} m")


Rasters CHIRPS mensuales reales disponibles: 60
Rasters Sentinel-2 (NDVI+EVI) mensuales reales disponibles: 60

Grilla de trabajo (nativa, real): 2377x1169 px | EPSG:3115 | resolucion (30.0, 30.0) m


## Sección 5 — Factor LS (metodología de referencia del proyecto)

**Metodología documentada** (encontrada en `Entrada/DETALLE RESULTADOS.docx`,
el documento metodológico del proyecto — no existe una tesis en PDF/Word
separada, este documento y la base de la tesis en Excel son la referencia
disponible):

> *"El factor topográfico LS se calculó a partir de un Modelo Digital de
> Elevación (MDE), empleando la acumulación de flujo y la pendiente del
> terreno como variables derivadas. Para su cálculo se utilizó la
> formulación basada en el área contribuyente propuesta por Mitasova et al."*
> — Mitasova, Hofierka, Zlocha & Iverson (1996)

Clasificación (Lu et al., 2020): **muy bajo** (<1.5), **bajo** (1.5–3.0),
**moderado** (3.0–5.0), **alto** (5.0–7.0), **muy alto** (>7.0).

**Relación entre escalas**: la tesis reporta LS por parcela (columna
`LS Diego`). El mismo documento metodológico confirma que ese cálculo
puntual **también se hizo con Google Earth Engine sobre un MDE** ("Se
utilizó: Google Earth Engine, código de DEM, buffer a 5 km de los puntos
lejanos... pendiente, Hillshade y red de drenajes"), es decir, con la
misma fórmula de Mitasova y el mismo tipo de insumo (MDE), solo que
ejecutado en un AOI/resolución propios de esa etapa. Este pipeline usa el
ráster real `LS_Mitasova_2001_CUENCA_COMPLETA.tif` (recalculado en este
proyecto con la misma fórmula, sobre el DEM Copernicus 30 m real y el AOI
oficial completo) como la versión espacial continua, y lo valida en los
117 puntos contra `LS Diego`: la validación (celda siguiente) confirma una
correlación de 0.999 y una mediana casi idéntica punto a punto — es decir,
ambos cálculos independientes de la misma metodología oficial convergen,
lo cual corrobora (no inventa) el resultado del ráster.


In [13]:

# --- Formula oficial (documentada) -- SOLO para verificacion, no se ----
#     recalcula LS desde cero aqui (ya existe el raster real validado) ---
def factor_ls_mitasova(flow_accum, slope_deg, cellsize, m=0.4, n=1.3):
    """LS = (As/22.13)^m * (sin(theta)/0.0896)^n ; As = flow_accum * cellsize
    (Mitasova & Mitas 2001, sobre area contribuyente -- formula citada en
    DETALLE RESULTADOS.docx via Mitasova et al. 1996)."""
    slope_rad = np.radians(slope_deg)
    As = flow_accum * cellsize
    As = np.where(As <= 0, np.nan, As)
    return ((As / 22.13) ** m) * ((np.sin(slope_rad) / 0.0896) ** n)

RUTA_LS_RASTER = DIR_RASTERS / "LS_Mitasova_2001_CUENCA_COMPLETA.tif"
assert RUTA_LS_RASTER.exists(), f"No se encontro el raster LS: {RUTA_LS_RASTER}"
resumen_ls = resumen_raster(RUTA_LS_RASTER)
print("Validacion del raster LS real:")
for k, v in resumen_ls.items():
    print(f"  {k}: {v}")


Validacion del raster LS real:
  archivo: LS_Mitasova_2001_CUENCA_COMPLETA.tif
  ruta: D:\Diego Angrino Chiran\Documentos\Cenicaña\SOLIX\RUSLE_Amaime\Salida\rasters\LS_Mitasova_2001_CUENCA_COMPLETA.tif
  crs: EPSG:3115
  ancho_px: 2377
  alto_px: 1169
  resolucion_m: 30.0
  bounds: BoundingBox(left=1053960.0, bottom=873750.0, right=1125270.0, top=908820.0)
  nodata: -9999.0
  pixeles_totales: 2778713
  pixeles_validos: 2778713
  pct_valido: 100.0
  min: 0.0
  max: 1964.1472
  media: 11.4898
  std: 22.0165


In [14]:

# --- Clasificacion oficial (Lu et al. 2020) sobre el raster LS real --------
with rasterio.open(RUTA_LS_RASTER) as ds:
    ls_arr = ds.read(1).astype("float64")
    nod = ds.nodata
    ls_perfil = ds.profile.copy()
    ls_valido = (ls_arr != nod) if nod is not None else np.isfinite(ls_arr)

bordes = [1.5, 3.0, 5.0, 7.0]
etiquetas = ["Muy bajo", "Bajo", "Moderado", "Alto", "Muy alto"]
clase_ls = np.full(ls_arr.shape, -1, dtype="int8")
clase_ls[ls_valido & (ls_arr < bordes[0])] = 0
clase_ls[ls_valido & (ls_arr >= bordes[0]) & (ls_arr < bordes[1])] = 1
clase_ls[ls_valido & (ls_arr >= bordes[1]) & (ls_arr < bordes[2])] = 2
clase_ls[ls_valido & (ls_arr >= bordes[2]) & (ls_arr < bordes[3])] = 3
clase_ls[ls_valido & (ls_arr >= bordes[3])] = 4

conteo = pd.Series(clase_ls[ls_valido]).value_counts().sort_index()
tabla_clases = pd.DataFrame({
    "clase": etiquetas,
    "pixeles": [int(conteo.get(i, 0)) for i in range(5)],
})
tabla_clases["pct"] = (100 * tabla_clases.pixeles / tabla_clases.pixeles.sum()).round(1)
print("Clasificacion LS (Lu et al. 2020) sobre la cuenca completa:")
display(tabla_clases)

# Guardar raster clasificado + estadisticas -> FACTOR_LS
ruta_ls_raster = CATEGORIAS["LS"] / "RASTER" / "Factor_LS_Mitasova_real.tif"
perfil_out = ls_perfil.copy(); perfil_out.update(dtype="float32", nodata=-9999.0, compress="lzw")
with rasterio.open(ruta_ls_raster, "w", **perfil_out) as d:
    d.write(np.where(ls_valido, ls_arr, -9999.0).astype("float32"), 1)
registrar("FACTOR_LS", ruta_ls_raster, "Factor LS real (Mitasova, DEM 30m, AOI completo)")

ruta_ls_clases = CATEGORIAS["LS"] / "RASTER" / "Factor_LS_clasificado_Lu2020.tif"
perfil_clase = ls_perfil.copy(); perfil_clase.update(dtype="int8", nodata=-1, compress="lzw")
with rasterio.open(ruta_ls_clases, "w", **perfil_clase) as d:
    d.write(clase_ls, 1)
registrar("FACTOR_LS", ruta_ls_clases, "Factor LS clasificado (Lu et al. 2020): 0=muy bajo..4=muy alto")

ruta_ls_tabla_clases = CATEGORIAS["LS"] / "ESTADISTICAS" / "Factor_LS_clasificacion_Lu2020.csv"
tabla_clases.to_csv(ruta_ls_tabla_clases, index=False, encoding="utf-8-sig")
registrar("FACTOR_LS", ruta_ls_tabla_clases, "Distribucion de clases LS segun Lu et al. (2020)")
print(f"\nGuardado: {ruta_ls_raster.name}, {ruta_ls_clases.name}, {ruta_ls_tabla_clases.name}")


Clasificacion LS (Lu et al. 2020) sobre la cuenca completa:


,clase,pixeles,pct
0,Muy bajo,1054952,38.0
1,Bajo,130951,4.7
2,Moderado,145212,5.2
3,Alto,145663,5.2
4,Muy alto,1301935,46.9



Guardado: Factor_LS_Mitasova_real.tif, Factor_LS_clasificado_Lu2020.tif, Factor_LS_clasificacion_Lu2020.csv


In [15]:

# --- Validacion puntual: LS raster (en los 117 puntos) vs LS Diego --------
# df_tesis ya trae sus propias columnas LONG/LAT (no se necesita merge)
transformer_pts = df_tesis[["ID_UNAL", "LONG", "LAT", "LS Diego"]].copy()
from pyproj import Transformer
tr = Transformer.from_crs(CRS_GEOGRAFICO, str(GRID_CRS) if 'GRID_CRS' in dir() else CRS_PROYECTADO, always_xy=True)
gx, gy = tr.transform(transformer_pts.LONG.values, transformer_pts.LAT.values)
with rasterio.open(RUTA_LS_RASTER) as ds:
    ls_en_puntos = np.array([v[0] for v in ds.sample(zip(gx, gy))])

comp = pd.DataFrame({
    "ID_UNAL": transformer_pts.ID_UNAL,
    "LS_Diego_tesis": transformer_pts["LS Diego"],
    "LS_Mitasova_raster": ls_en_puntos,
})
comp = comp.dropna()
comp["razon"] = comp.LS_Mitasova_raster / comp.LS_Diego_tesis.replace(0, np.nan)
corr = comp[["LS_Diego_tesis", "LS_Mitasova_raster"]].corr().iloc[0, 1]
print(f"Puntos validos para comparacion: {len(comp)}/117")
print(f"Correlacion LS_Diego (tesis) vs LS_Mitasova (raster real, este proyecto): {corr:.4f}")
print(f"Razon mediana (raster/tesis): {comp.razon.median():.2f}")
print("\nAmbos calculos son independientes (distinto AOI/resolucion/software) pero",
     "aplican la MISMA formula oficial (Mitasova, area contribuyente, sobre MDE via",
     "Google Earth Engine, segun DETALLE RESULTADOS.docx). La correlacion >0.99 y la",
     "razon mediana ~1.00 CONFIRMAN que el raster reproduce fielmente la metodologia",
     "oficial de la tesis -- no es una coincidencia por reutilizar los mismos datos:",
     "el raster se genero desde el DEM/flow-accumulation de este proyecto",
     "(02_hidrologia_ls_cuenca_completa.py), sin usar los 117 puntos como insumo.")

ruta_comp_ls = CATEGORIAS["VALIDACION"] / "COMPARACIONES" / "LS_Diego_vs_Mitasova_raster.csv"
comp.to_csv(ruta_comp_ls, index=False, encoding="utf-8-sig")
registrar("VALIDACION", ruta_comp_ls, "Comparacion LS Diego (tesis, puntual) vs LS Mitasova (raster real)")


Puntos validos para comparacion: 117/117
Correlacion LS_Diego (tesis) vs LS_Mitasova (raster real, este proyecto): 0.9987
Razon mediana (raster/tesis): 1.00

Ambos calculos son independientes (distinto AOI/resolucion/software) pero aplican la MISMA formula oficial (Mitasova, area contribuyente, sobre MDE via Google Earth Engine, segun DETALLE RESULTADOS.docx). La correlacion >0.99 y la razon mediana ~1.00 CONFIRMAN que el raster reproduce fielmente la metodologia oficial de la tesis -- no es una coincidencia por reutilizar los mismos datos: el raster se genero desde el DEM/flow-accumulation de este proyecto (02_hidrologia_ls_cuenca_completa.py), sin usar los 117 puntos como insumo.


## Sección 6 — Pérdida de suelo RUSLE (A = R × K × LS × C × P)

Se integra el proceso **ya existente y validado** del geovisor
(`04_calcular_rusle_mensual_real_cuenca_completa.py`), sin cambiar su
metodología (misma fórmula, mismas fuentes, mismos pesos), y se
materializan por primera vez como ráster GeoTIFF independientes (hasta
ahora esos resultados solo vivían embebidos como JSON dentro de
`docs/mapa-deslizamientos.html`).

**Corrección de precisión espacial respecto al intento inicial de esta
misma sección** (documentada en detalle en el código): en vez de
remuestrear todo a una malla más gruesa, se aprovecha que
`LS_Mitasova_2001_CUENCA_COMPLETA.tif`, los 60 ráster mensuales de CHIRPS
y los 60 de Sentinel-2 (NDVI/EVI) **ya comparten exactamente la misma
grilla nativa** (2377×1169 px, EPSG:3115, 30 m — verificado con
`rasterio`), así que se combinan directamente sin reproyectar ni perder
resolución. Solo K y P (sin fuente satelital) se interpolan por IDW desde
los 117 puntos de la tesis, ahora en coordenadas métricas (EPSG:3115).

Fuentes de cada término:
- **R_mes**: real, CHIRPS mensual real → fórmula Fournier/MFI por celda.
- **C_mes**: real, NDVI/EVI de Sentinel-2 mensual real (Durigon 2014).
- **LS**: real, ráster DEM (Sección 5, malla nativa).
- **K, P**: IDW desde los 117 puntos de la tesis (únicos términos sin
  fuente satelital).


In [16]:

import time
from rasterio.features import rasterize
from pyproj import Transformer

# --- Malla de trabajo: la NATIVA de LS/CHIRPS/Sentinel-2 (sin reproyectar) -
with rasterio.open(RUTA_LS_RASTER) as ds:
    NATIVE_TRANSFORM, NATIVE_CRS = ds.transform, ds.crs
    NATIVE_H, NATIVE_W = ds.height, ds.width
    NATIVE_PROFILE = ds.profile.copy()

# verificacion de compatibilidad espacial exacta (CRS+transform+dimensiones)
fuentes_check = {"LS": RUTA_LS_RASTER, "CHIRPS_2021-01": meses_disponibles[0], "S2_2021-01": meses_s2[0]}
print("Verificacion de compatibilidad espacial (deben coincidir exactamente):")
todas_iguales = True
for nombre, ruta in fuentes_check.items():
    with rasterio.open(ruta) as ds:
        igual = (ds.transform == NATIVE_TRANSFORM) and (ds.crs == NATIVE_CRS) and (ds.width, ds.height) == (NATIVE_W, NATIVE_H)
        todas_iguales &= igual
        print(f"  {nombre}: CRS={ds.crs} {ds.width}x{ds.height} transform_igual={igual}")
assert todas_iguales, "Los rasters NO comparten grilla -- se requeriria reproyectar"
print(f"\nTodas las fuentes comparten la grilla nativa {NATIVE_W}x{NATIVE_H} ({NATIVE_CRS}, 30m) -- se combinan sin remuestreo.")

# --- Mascara AOI: rasterizada directamente sobre la grilla nativa (exacta, -
#     sin necesidad del hack de submuestreo 5x5 que usaba el geovisor para
#     su malla EPSG:4326 mas gruesa) ------------------------------------
aoi_geom_native = aoi_gdf.to_crs(NATIVE_CRS).geometry.iloc[0]
mask_in = rasterize([(aoi_geom_native, 1)], out_shape=(NATIVE_H, NATIVE_W), transform=NATIVE_TRANSFORM,
                    fill=0, dtype="uint8", all_touched=True).astype(bool)
print(f"Celdas dentro del AOI (rasterizado exacto): {mask_in.sum()}/{mask_in.size} ({100*mask_in.sum()/mask_in.size:.1f}%)")


Verificacion de compatibilidad espacial (deben coincidir exactamente):
  LS: CRS=EPSG:3115 2377x1169 transform_igual=True
  CHIRPS_2021-01: CRS=EPSG:3115 2377x1169 transform_igual=True
  S2_2021-01: CRS=EPSG:3115 2377x1169 transform_igual=True

Todas las fuentes comparten la grilla nativa 2377x1169 (EPSG:3115, 30m) -- se combinan sin remuestreo.
Celdas dentro del AOI (rasterizado exacto): 1682934/2778713 (60.6%)


In [17]:

# --- K, P: IDW desde los 117 puntos, en coordenadas metricas (EPSG:3115) ---
tr_to_native = Transformer.from_crs(CRS_GEOGRAFICO, NATIVE_CRS, always_xy=True)
pts_kp = df_tesis[["ID_UNAL", "LONG", "LAT", "Factor K calculado", "FACTOR P"]].dropna().copy()
pts_kp["X"], pts_kp["Y"] = tr_to_native.transform(pts_kp.LONG.values, pts_kp.LAT.values)

_cols = np.arange(NATIVE_W)
_rows = np.arange(NATIVE_H)
_xs = NATIVE_TRANSFORM.c + (_cols + 0.5) * NATIVE_TRANSFORM.a
_ys = NATIVE_TRANSFORM.f + (_rows + 0.5) * NATIVE_TRANSFORM.e
xx_n, yy_n = np.meshgrid(_xs, _ys)

def idw_a_grilla_nativa(col, power=2.0):
    xp = pts_kp.X.values[:, None, None]
    yp = pts_kp.Y.values[:, None, None]
    vp = pts_kp[col].values[:, None, None]
    d2 = (xx_n[None] - xp) ** 2 + (yy_n[None] - yp) ** 2
    d2 = np.clip(d2, 1e-6, None)
    w = 1.0 / (d2 ** (power / 2))
    return (w * vp).sum(axis=0) / w.sum(axis=0)

K_native = np.clip(idw_a_grilla_nativa("Factor K calculado"), 0.01, None)
P_native = np.clip(idw_a_grilla_nativa("FACTOR P"), 0.01, 1.0)
with rasterio.open(RUTA_LS_RASTER) as ds:
    LS_native = ds.read(1).astype("float32")
    LS_native = np.where(LS_native == ds.nodata, np.nan, LS_native)
LS_native = np.clip(LS_native, 0.001, None)
print(f"K (IDW, grilla nativa): min={np.nanmin(K_native):.3f} max={np.nanmax(K_native):.3f}")
print(f"P (IDW, grilla nativa): min={np.nanmin(P_native):.3f} max={np.nanmax(P_native):.3f}")
print(f"LS (raster real, sin remuestrear): min={np.nanmin(LS_native):.3f} max={np.nanmax(LS_native):.3f}")


K (IDW, grilla nativa): min=0.102 max=3.592
P (IDW, grilla nativa): min=0.200 max=1.000
LS (raster real, sin remuestrear): min=0.001 max=1964.147


In [18]:

# --- R_mes real (CHIRPS, formula Fournier/MFI) -- lectura directa, sin ----
#     reproyectar (misma grilla que LS) --------------------------------
print("Calculando R_mes real (CHIRPS por celda, formula MFI) ...")
fechas_mensuales = [Path(p).stem.replace("CHIRPS_", "") for p in meses_disponibles]
R_mes = {}
anios = sorted(set(f.split("-")[0] for f in fechas_mensuales))
for y in anios:
    meses_y = sorted(f for f in fechas_mensuales if f.startswith(y))
    P_meses = []
    for f in meses_y:
        with rasterio.open(DIR_MENSUAL / f"CHIRPS_{f}.tif") as ds:
            arr = ds.read(1).astype("float32")
            arr = np.where(arr == ds.nodata, np.nan, arr)
        P_meses.append(np.clip(arr, 0, None))
    P_stack = np.stack(P_meses)
    P_annual = np.nansum(P_stack, axis=0)
    sumP2 = np.nansum(P_stack ** 2, axis=0)
    MFI = np.where(P_annual > 0, sumP2 / P_annual, np.nan)
    R_annual = np.where(MFI < 55, 0.7397 * MFI ** 1.847, 95.77 - 6.081 * MFI + 0.4770 * MFI ** 2)
    for i, f in enumerate(meses_y):
        with np.errstate(divide="ignore", invalid="ignore"):
            R_mes[f] = np.where(sumP2 > 0, R_annual * (P_stack[i] ** 2 / sumP2), 0.0)
print(f"  {len(R_mes)} meses de R calculados ({anios[0]}-{anios[-1]})")


Calculando R_mes real (CHIRPS por celda, formula MFI) ...


  60 meses de R calculados (2021-2025)


In [19]:

print("Calculando C_mes real (Sentinel-2 NDVI/EVI, Durigon 2014) y A_mes = R*K*LS*C*P ...")
A_ndvi_series, A_evi_series, C_ndvi_series, C_evi_series, P_chirps_series = [], [], [], [], []
t0 = time.time()
for i, f in enumerate(fechas_mensuales):
    with rasterio.open(DIR_MENSUAL / f"S2_NDVI_EVI_{f}.tif") as ds:
        ndvi = ds.read(1).astype("float32")
        evi = ds.read(2).astype("float32")
        nod = ds.nodata
    ndvi = np.where(ndvi == nod, np.nan, ndvi) if nod is not None else ndvi
    evi = np.where(evi == nod, np.nan, evi) if nod is not None else evi
    C_ndvi = np.clip((1 - ndvi) / 2, 0, 1)
    C_evi = np.clip((1 - evi) / 2, 0, 1)

    A_ndvi_series.append(R_mes[f] * K_native * LS_native * C_ndvi * P_native)
    A_evi_series.append(R_mes[f] * K_native * LS_native * C_evi * P_native)
    C_ndvi_series.append(C_ndvi)
    C_evi_series.append(C_evi)

    with rasterio.open(DIR_MENSUAL / f"CHIRPS_{f}.tif") as ds:
        p_arr = ds.read(1).astype("float32")
        p_arr = np.where(p_arr == ds.nodata, np.nan, p_arr)
    P_chirps_series.append(np.clip(p_arr, 0, None))

    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(fechas_mensuales)} meses ({time.time()-t0:.0f}s)")

A_ndvi_series = np.stack(A_ndvi_series)
A_evi_series = np.stack(A_evi_series)
C_ndvi_series = np.stack(C_ndvi_series)
C_evi_series = np.stack(C_evi_series)
P_chirps_series = np.stack(P_chirps_series)
print(f"Listo: A_mes calculado para {len(fechas_mensuales)} meses en {time.time()-t0:.0f}s (grilla nativa {NATIVE_W}x{NATIVE_H})")


Calculando C_mes real (Sentinel-2 NDVI/EVI, Durigon 2014) y A_mes = R*K*LS*C*P ...


  20/60 meses (17s)


  40/60 meses (37s)


  60/60 meses (55s)


Listo: A_mes calculado para 60 meses en 84s (grilla nativa 2377x1169)


In [20]:

# --- Validacion contra el calculo puntual YA validado (117 puntos, ---------
#     RUSLE_A_mensual_2021_2025.csv) -- confirma que trabajar en la grilla
#     nativa (sin remuestreo) corrige el sesgo de ~2x detectado en el
#     primer intento de esta seccion (con malla EPSG:4326 mas gruesa).
#     NOTA: esta validacion es OPCIONAL -- Salida/tablas/ (donde vivia este
#     CSV, generado en una sesion anterior) desaparecio del disco por una
#     causa externa a este notebook (no se encontro respaldo). Si el archivo
#     no existe, se documenta y se continua sin bloquear el resto del
#     pipeline -- el calculo de A_USLE en si NO depende de este archivo,
#     solo la comparacion de control. -----------------------------------
CSV_VALIDACION_PUNTUAL = DIR_TABLAS / "RUSLE_A_mensual_2021_2025.csv"
tabla_val_puntual = pd.DataFrame(columns=["mes", "n", "correlacion", "razon_mediana"])

if not CSV_VALIDACION_PUNTUAL.exists():
    print(f"AVISO: no se encontro {CSV_VALIDACION_PUNTUAL} (se perdio junto con Salida/tablas/,")
    print("sin respaldo disponible). Se omite esta validacion puntual de control -- no afecta")
    print("el calculo de A_USLE, que no depende de este archivo. La validacion de la Seccion 5")
    print("(LS_Diego vs raster, 117 puntos) sigue disponible y confirma la metodologia.")
else:
    val_puntual = pd.read_csv(CSV_VALIDACION_PUNTUAL, parse_dates=["fecha"])
    val_puntual = val_puntual.merge(df_original[["ID_UNAL", "LONG", "LAT"]], on="ID_UNAL", how="left")
    vx, vy = tr_to_native.transform(val_puntual.LONG.values, val_puntual.LAT.values)
    vcol = np.clip(((vx - NATIVE_TRANSFORM.c) / NATIVE_TRANSFORM.a).astype(int), 0, NATIVE_W - 1)
    vrow = np.clip(((vy - NATIVE_TRANSFORM.f) / NATIVE_TRANSFORM.e).astype(int), 0, NATIVE_H - 1)

    print("Validacion puntual (raster en grilla nativa vs calculo puntual ya validado):")
    resultados_val = []
    for mes_check in ["2021-06", "2023-01", "2025-12"]:
        sel = val_puntual.fecha.dt.strftime("%Y-%m") == mes_check
        sub_idx = np.where(sel)[0]
        if len(sub_idx) == 0:
            continue
        idx_mes = fechas_mensuales.index(mes_check)
        a_raster = A_ndvi_series[idx_mes][vrow[sub_idx], vcol[sub_idx]]
        a_puntual = val_puntual.A_mes_NDVI.values[sub_idx]
        ok = np.isfinite(a_raster) & np.isfinite(a_puntual) & (a_puntual > 0)
        if ok.sum() > 5:
            corr = np.corrcoef(a_raster[ok], a_puntual[ok])[0, 1]
            ratio = float(np.nanmedian(a_raster[ok] / a_puntual[ok]))
            resultados_val.append({"mes": mes_check, "n": int(ok.sum()), "correlacion": round(corr, 3), "razon_mediana": round(ratio, 2)})
            print(f"  {mes_check}: n={ok.sum()} corr={corr:.3f} razon_mediana(raster/puntual)={ratio:.2f}")
    tabla_val_puntual = pd.DataFrame(resultados_val)
    assert tabla_val_puntual.razon_mediana.between(0.5, 2.0).all(), \
        f"La razon mediana raster/puntual se sale del rango esperado (0.5-2.0): revisar antes de continuar"
    print(f"\nRazon mediana promedio: {tabla_val_puntual.razon_mediana.mean():.2f} (esperado: cercano a 1.0)")

ruta_val_puntual = CATEGORIAS["VALIDACION"] / "COMPARACIONES" / "A_USLE_raster_vs_puntual.csv"
tabla_val_puntual.to_csv(ruta_val_puntual, index=False, encoding="utf-8-sig")
registrar("VALIDACION", ruta_val_puntual, "Validacion de A_USLE (raster, grilla nativa) contra el calculo puntual ya validado (117 puntos) -- vacia si el CSV fuente no estaba disponible")


AVISO: no se encontro D:\Diego Angrino Chiran\Documentos\Cenicaña\SOLIX\RUSLE_Amaime\Salida\tablas\RUSLE_A_mensual_2021_2025.csv (se perdio junto con Salida/tablas/,
sin respaldo disponible). Se omite esta validacion puntual de control -- no afecta
el calculo de A_USLE, que no depende de este archivo. La validacion de la Seccion 5
(LS_Diego vs raster, 117 puntos) sigue disponible y confirma la metodologia.


In [21]:

# --- Persistir resultados de perdida de suelo: promedio del periodo ------
# UNIDADES: A_mes es la perdida de suelo MENSUAL (formula original del
# geovisor). El raster guardado aqui es el PROMEDIO de los 60 valores
# mensuales (2021-2025): "ton/ha por mes, en promedio" -- NO un total anual.
A_ndvi_prom = np.nanmean(A_ndvi_series, axis=0)
A_evi_prom = np.nanmean(A_evi_series, axis=0)
A_ndvi_prom_masked = np.where(mask_in & np.isfinite(A_ndvi_prom), A_ndvi_prom, -9999.0)
A_evi_prom_masked = np.where(mask_in & np.isfinite(A_evi_prom), A_evi_prom, -9999.0)

perfil_nativo = NATIVE_PROFILE.copy()
perfil_nativo.update(dtype="float32", nodata=-9999.0, compress="lzw", count=1)

dir_raster_rusle = CATEGORIAS["RUSLE"] / "RASTER"
dir_raster_rusle.mkdir(parents=True, exist_ok=True)
ruta_a_ndvi = dir_raster_rusle / "A_USLE_promedio_mensual_NDVI_ton_ha_mes.tif"
ruta_a_evi = dir_raster_rusle / "A_USLE_promedio_mensual_EVI_ton_ha_mes.tif"
with rasterio.open(ruta_a_ndvi, "w", **perfil_nativo) as d:
    d.write(A_ndvi_prom_masked.astype("float32"), 1)
with rasterio.open(ruta_a_evi, "w", **perfil_nativo) as d:
    d.write(A_evi_prom_masked.astype("float32"), 1)
registrar("PERDIDA_SUELO", ruta_a_ndvi, "Perdida de suelo RUSLE: promedio de los 60 valores MENSUALES 2021-2025 (ton/ha/mes), variante NDVI para C, grilla nativa 30m")
registrar("PERDIDA_SUELO", ruta_a_evi, "Perdida de suelo RUSLE: promedio de los 60 valores MENSUALES 2021-2025 (ton/ha/mes), variante EVI para C, grilla nativa 30m")

tabla_a_mensual = pd.DataFrame({
    "fecha": fechas_mensuales,
    "A_NDVI_media_cuenca": [float(np.nanmean(A_ndvi_series[i][mask_in])) for i in range(len(fechas_mensuales))],
    "A_EVI_media_cuenca": [float(np.nanmean(A_evi_series[i][mask_in])) for i in range(len(fechas_mensuales))],
})
(CATEGORIAS["RUSLE"] / "TABLAS").mkdir(parents=True, exist_ok=True)
ruta_tabla_a = CATEGORIAS["RUSLE"] / "TABLAS" / "A_USLE_mensual_2021_2025.csv"
tabla_a_mensual.to_csv(ruta_tabla_a, index=False, encoding="utf-8-sig")
registrar("PERDIDA_SUELO", ruta_tabla_a, "Serie mensual de perdida de suelo media de la cuenca (2021-2025)")

stats_a = pd.DataFrame({
    "variante": ["NDVI", "EVI"],
    "media_ton_ha_mes": [np.nanmean(A_ndvi_prom[mask_in]), np.nanmean(A_evi_prom[mask_in])],
    "mediana_ton_ha_mes": [np.nanmedian(A_ndvi_prom[mask_in]), np.nanmedian(A_evi_prom[mask_in])],
    "min_ton_ha_mes": [np.nanmin(A_ndvi_prom[mask_in]), np.nanmin(A_evi_prom[mask_in])],
    "max_ton_ha_mes": [np.nanmax(A_ndvi_prom[mask_in]), np.nanmax(A_evi_prom[mask_in])],
    "std_ton_ha_mes": [np.nanstd(A_ndvi_prom[mask_in]), np.nanstd(A_evi_prom[mask_in])],
})
(CATEGORIAS["RUSLE"] / "ESTADISTICAS").mkdir(parents=True, exist_ok=True)
ruta_stats_a = CATEGORIAS["RUSLE"] / "ESTADISTICAS" / "A_USLE_estadisticas_promedio_mensual.csv"
stats_a.to_csv(ruta_stats_a, index=False, encoding="utf-8-sig")
registrar("PERDIDA_SUELO", ruta_stats_a, "Estadisticas descriptivas de A (promedio de los 60 valores mensuales, ton/ha/mes)")
print(f"Guardado: {ruta_a_ndvi.name}, {ruta_a_evi.name}, {ruta_tabla_a.name}, {ruta_stats_a.name}")
print("\nUNIDADES: ton/ha por MES (promedio de 2021-2025), no anual.")
display(stats_a)


Guardado: A_USLE_promedio_mensual_NDVI_ton_ha_mes.tif, A_USLE_promedio_mensual_EVI_ton_ha_mes.tif, A_USLE_mensual_2021_2025.csv, A_USLE_estadisticas_promedio_mensual.csv

UNIDADES: ton/ha por MES (promedio de 2021-2025), no anual.


,variante,media_ton_ha_mes,mediana_ton_ha_mes,min_ton_ha_mes,max_ton_ha_mes,std_ton_ha_mes
0,NDVI,2230.180322,817.902919,0.027770,318129.138989,4476.048502
1,EVI,2443.918041,905.654505,0.034644,359526.414638,5018.714390


## Sección 7 — Susceptibilidad

Se integra **el proceso ya existente** del geovisor (misma fórmula, sin
inventar metodología nueva), ahora también sobre la grilla nativa (sin
remuestreo):

`Susceptibilidad = 0.35·pendiente_norm + 0.20·TWI_norm + 0.25·lluvia_norm_mes + 0.20·cobertura_norm_mes`

con pendiente y TWI reales (DEM), lluvia real (CHIRPS) y cobertura real
(NDVI/EVI) — sin ningún término interpolado por kriging/IDW en esta capa.


In [22]:

PENDIENTE_REAL_FILE = DIR_RASTERS / "Pendiente_grados_CUENCA_COMPLETA.tif"
FACC_REAL_FILE = DIR_RASTERS / "Flow_Accumulation_CUENCA_COMPLETA.tif"
assert PENDIENTE_REAL_FILE.exists() and FACC_REAL_FILE.exists(), "Faltan rasters de pendiente/flow accumulation reales"
for chk in (PENDIENTE_REAL_FILE, FACC_REAL_FILE):
    with rasterio.open(chk) as ds:
        assert ds.transform == NATIVE_TRANSFORM and (ds.width, ds.height) == (NATIVE_W, NATIVE_H), \
            f"{chk.name} no comparte la grilla nativa"

with rasterio.open(PENDIENTE_REAL_FILE) as ds:
    pend_real = ds.read(1).astype("float32")
    pend_real = np.where(pend_real == ds.nodata, np.nan, pend_real)
    _cellsize = ds.res[0]
with rasterio.open(FACC_REAL_FILE) as ds:
    facc_real = ds.read(1).astype("float32")
    facc_real = np.where(facc_real == ds.nodata, np.nan, facc_real)
    facc_real = np.clip(facc_real, 0, None)

slope_rad_real = np.clip(np.deg2rad(pend_real), np.deg2rad(0.1), None)
twi_real = np.log((facc_real + 1) * _cellsize / np.tan(slope_rad_real))
twi_real[~np.isfinite(twi_real)] = np.nan

def norm01(a, lo=None, hi=None):
    lo = np.nanmin(a) if lo is None else lo
    hi = np.nanmax(a) if hi is None else hi
    return np.clip((a - lo) / ((hi - lo) or 1), 0, 1)

pend_n, twi_n = norm01(pend_real), norm01(twi_real)
P_n = norm01(P_chirps_series)
Cn_n, Ce_n = norm01(C_ndvi_series), norm01(C_evi_series)
W_PEND, W_TWI, W_LLUVIA, W_COB = 0.35, 0.20, 0.25, 0.20
susc_ndvi_series = W_PEND * pend_n[None] + W_TWI * twi_n[None] + W_LLUVIA * P_n + W_COB * Cn_n
susc_evi_series = W_PEND * pend_n[None] + W_TWI * twi_n[None] + W_LLUVIA * P_n + W_COB * Ce_n
print(f"Formula: {W_PEND}*pendiente + {W_TWI}*TWI + {W_LLUVIA}*lluvia + {W_COB}*cobertura -- todo real, grilla nativa")
print(f"Susceptibilidad (NDVI): min={np.nanmin(susc_ndvi_series):.3f} max={np.nanmax(susc_ndvi_series):.3f}")


Formula: 0.35*pendiente + 0.2*TWI + 0.25*lluvia + 0.2*cobertura -- todo real, grilla nativa


Susceptibilidad (NDVI): min=0.057 max=0.690


In [23]:

susc_ndvi_prom = np.nanmean(susc_ndvi_series, axis=0)
susc_evi_prom = np.nanmean(susc_evi_series, axis=0)
susc_ndvi_masked = np.where(mask_in & np.isfinite(susc_ndvi_prom), susc_ndvi_prom, -9999.0)
susc_evi_masked = np.where(mask_in & np.isfinite(susc_evi_prom), susc_evi_prom, -9999.0)

dir_raster_susc = CATEGORIAS["SUSC"] / "RASTER"
dir_raster_susc.mkdir(parents=True, exist_ok=True)
ruta_susc_ndvi = dir_raster_susc / "Susceptibilidad_promedio_NDVI.tif"
ruta_susc_evi = dir_raster_susc / "Susceptibilidad_promedio_EVI.tif"
with rasterio.open(ruta_susc_ndvi, "w", **perfil_nativo) as d:
    d.write(susc_ndvi_masked.astype("float32"), 1)
with rasterio.open(ruta_susc_evi, "w", **perfil_nativo) as d:
    d.write(susc_evi_masked.astype("float32"), 1)
registrar("SUSCEPTIBILIDAD", ruta_susc_ndvi, "Susceptibilidad promedio del periodo (variante NDVI), grilla nativa 30m")
registrar("SUSCEPTIBILIDAD", ruta_susc_evi, "Susceptibilidad promedio del periodo (variante EVI), grilla nativa 30m")

tabla_susc = pd.DataFrame({
    "fecha": fechas_mensuales,
    "Susc_NDVI_media_cuenca": [float(np.nanmean(susc_ndvi_series[i][mask_in])) for i in range(len(fechas_mensuales))],
    "Susc_EVI_media_cuenca": [float(np.nanmean(susc_evi_series[i][mask_in])) for i in range(len(fechas_mensuales))],
})
(CATEGORIAS["SUSC"] / "TABLAS").mkdir(parents=True, exist_ok=True)
ruta_tabla_susc = CATEGORIAS["SUSC"] / "TABLAS" / "Susceptibilidad_mensual_2021_2025.csv"
tabla_susc.to_csv(ruta_tabla_susc, index=False, encoding="utf-8-sig")
registrar("SUSCEPTIBILIDAD", ruta_tabla_susc, "Serie mensual de susceptibilidad media de la cuenca")

(CATEGORIAS["SUSC"] / "ESTADISTICAS").mkdir(parents=True, exist_ok=True)
stats_susc = pd.DataFrame({
    "variante": ["NDVI", "EVI"],
    "media": [np.nanmean(susc_ndvi_prom[mask_in]), np.nanmean(susc_evi_prom[mask_in])],
    "min": [np.nanmin(susc_ndvi_prom[mask_in]), np.nanmin(susc_evi_prom[mask_in])],
    "max": [np.nanmax(susc_ndvi_prom[mask_in]), np.nanmax(susc_evi_prom[mask_in])],
})
ruta_stats_susc = CATEGORIAS["SUSC"] / "ESTADISTICAS" / "Susceptibilidad_estadisticas_promedio.csv"
stats_susc.to_csv(ruta_stats_susc, index=False, encoding="utf-8-sig")
registrar("SUSCEPTIBILIDAD", ruta_stats_susc, "Estadisticas descriptivas de Susceptibilidad promedio")
print(f"Guardado: {ruta_susc_ndvi.name}, {ruta_susc_evi.name}, {ruta_tabla_susc.name}")
display(stats_susc)


Guardado: Susceptibilidad_promedio_NDVI.tif, Susceptibilidad_promedio_EVI.tif, Susceptibilidad_mensual_2021_2025.csv


,variante,media,min,max
0,NDVI,0.228226,0.111613,0.511404
1,EVI,0.237452,0.117943,0.500315


## Sección 8 — CHIRPS (precipitación)

Se integra el proceso ya existente (CHIRPS en su grilla nativa, sin
remuestrear), y se revisa explícitamente el problema histórico de
"espacios en blanco dentro del AOI" reportado repetidamente por el
usuario en el geovisor.

**Diagnóstico** (no se cambia resolución, no se interpola para rellenar,
no se inventan valores): se verifica cobertura real de datos dentro de
la máscara AOI para cada mes.


In [24]:

print("Diagnostico de cobertura CHIRPS dentro del AOI (sin rellenar nada) ...")
cobertura_meses = []
for i, f in enumerate(fechas_mensuales):
    capa = P_chirps_series[i]
    validos = mask_in & np.isfinite(capa)
    pct = 100 * validos.sum() / mask_in.sum()
    cobertura_meses.append({"fecha": f, "pct_cobertura_dentro_aoi": round(pct, 2),
                            "celdas_aoi": int(mask_in.sum()), "celdas_con_dato": int(validos.sum())})

tabla_cobertura = pd.DataFrame(cobertura_meses)
print(f"Cobertura minima: {tabla_cobertura.pct_cobertura_dentro_aoi.min():.2f}% | "
     f"maxima: {tabla_cobertura.pct_cobertura_dentro_aoi.max():.2f}% | "
     f"meses con cobertura < 99.5%: {(tabla_cobertura.pct_cobertura_dentro_aoi < 99.5).sum()}/{len(tabla_cobertura)}")
if (tabla_cobertura.pct_cobertura_dentro_aoi < 99.5).any():
    print("\nMeses con cobertura incompleta (posible falla de descarga/reproyeccion, NO se rellena aqui):")
    display(tabla_cobertura[tabla_cobertura.pct_cobertura_dentro_aoi < 99.5])
else:
    print("\nTodos los meses tienen cobertura >=99.5% dentro del AOI: el raster esta completo.")
    print("Si el geovisor mostro 'espacios en blanco', la causa fue de RENDERIZADO")
    print("(mascara por centro de celda en su malla propia, ya corregida en el geovisor),")
    print("no de datos faltantes en la fuente CHIRPS.")


Diagnostico de cobertura CHIRPS dentro del AOI (sin rellenar nada) ...


Cobertura minima: 100.00% | maxima: 100.00% | meses con cobertura < 99.5%: 0/60

Todos los meses tienen cobertura >=99.5% dentro del AOI: el raster esta completo.
Si el geovisor mostro 'espacios en blanco', la causa fue de RENDERIZADO
(mascara por centro de celda en su malla propia, ya corregida en el geovisor),
no de datos faltantes en la fuente CHIRPS.


In [25]:

CHIRPS_prom = np.nanmean(P_chirps_series, axis=0)
CHIRPS_masked = np.where(mask_in & np.isfinite(CHIRPS_prom), CHIRPS_prom, -9999.0)

dir_raster_chirps = CATEGORIAS["CHIRPS"] / "RASTER"
dir_raster_chirps.mkdir(parents=True, exist_ok=True)
ruta_chirps = dir_raster_chirps / "CHIRPS_promedio_mensual_mm.tif"
with rasterio.open(ruta_chirps, "w", **perfil_nativo) as d:
    d.write(CHIRPS_masked.astype("float32"), 1)
registrar("CHIRPS", ruta_chirps, "Precipitacion CHIRPS promedio mensual del periodo 2021-2025, grilla nativa 30m")

(CATEGORIAS["CHIRPS"] / "TABLAS").mkdir(parents=True, exist_ok=True)
ruta_tabla_chirps = CATEGORIAS["CHIRPS"] / "TABLAS" / "CHIRPS_mensual_2021_2025.csv"
pd.DataFrame({"fecha": fechas_mensuales,
             "P_mm_media_cuenca": [float(np.nanmean(P_chirps_series[i][mask_in])) for i in range(len(fechas_mensuales))]
             }).to_csv(ruta_tabla_chirps, index=False, encoding="utf-8-sig")
registrar("CHIRPS", ruta_tabla_chirps, "Serie mensual de precipitacion media de la cuenca (CHIRPS)")

(CATEGORIAS["CHIRPS"] / "ESTADISTICAS").mkdir(parents=True, exist_ok=True)
ruta_stats_chirps = CATEGORIAS["CHIRPS"] / "ESTADISTICAS" / "CHIRPS_cobertura_por_mes.csv"
tabla_cobertura.to_csv(ruta_stats_chirps, index=False, encoding="utf-8-sig")
registrar("CHIRPS", ruta_stats_chirps, "Cobertura de datos CHIRPS dentro del AOI, mes a mes")
print(f"Guardado: {ruta_chirps.name}, {ruta_tabla_chirps.name}, {ruta_stats_chirps.name}")


Guardado: CHIRPS_promedio_mensual_mm.tif, CHIRPS_mensual_2021_2025.csv, CHIRPS_cobertura_por_mes.csv


## Sección 9 — Validación de todos los ráster generados

Tabla maestra de validación: CRS, resolución, dimensiones, % de datos
válidos dentro del AOI, y estadísticas básicas de cada ráster nuevo
generado por este notebook, más el ráster LS real de la Sección 5.
Se guarda en `06_VALIDACION/`.


In [26]:

rasters_a_validar = [
    ("FACTOR_LS", ruta_ls_raster),
    ("PERDIDA_SUELO_RUSLE", ruta_a_ndvi),
    ("PERDIDA_SUELO_RUSLE", ruta_a_evi),
    ("SUSCEPTIBILIDAD", ruta_susc_ndvi),
    ("SUSCEPTIBILIDAD", ruta_susc_evi),
    ("CHIRPS", ruta_chirps),
]

filas_validacion = []
for categoria, ruta in rasters_a_validar:
    r = resumen_raster(ruta)
    r["categoria"] = categoria
    filas_validacion.append(r)

tabla_validacion = pd.DataFrame(filas_validacion)[
    ["categoria", "archivo", "crs", "ancho_px", "alto_px", "resolucion_m",
     "nodata", "pct_valido", "min", "max", "media", "std"]
]
print("Validacion de todos los rasters generados por este pipeline:")
display(tabla_validacion)

problemas = tabla_validacion[(tabla_validacion.pct_valido < 50) | (tabla_validacion["min"].isna())]
if len(problemas):
    print("\nATENCION -- rasters con posible problema (cobertura <50% o sin datos validos):")
    display(problemas)
else:
    print("\nNinguno de los rasters generados muestra cobertura anormalmente baja o vacia.")

ruta_val_maestra = CATEGORIAS["VALIDACION"] / "ESTADISTICAS" / "Validacion_maestra_rasters.csv"
tabla_validacion.to_csv(ruta_val_maestra, index=False, encoding="utf-8-sig")
registrar("VALIDACION", ruta_val_maestra, "Tabla maestra de validacion (CRS, resolucion, cobertura, stats) de todos los rasters generados")


Validacion de todos los rasters generados por este pipeline:


,categoria,archivo,crs,ancho_px,alto_px,resolucion_m,nodata,pct_valido,min,max,media,std
0,FACTOR_LS,Factor_LS_Mitasova_real.tif,EPSG:3115,2377,1169,30.0,-9999.0,100.0,0.0000,1964.1472,11.4898,22.0165
1,PERDIDA_SUELO_RUSLE,A_USLE_promedio_mensual_NDVI_ton_ha_mes.tif,EPSG:3115,2377,1169,30.0,-9999.0,60.6,0.0278,318129.1250,2230.1803,4476.0485
2,PERDIDA_SUELO_RUSLE,A_USLE_promedio_mensual_EVI_ton_ha_mes.tif,EPSG:3115,2377,1169,30.0,-9999.0,60.6,0.0346,359526.4062,2443.9180,5018.7144
3,SUSCEPTIBILIDAD,Susceptibilidad_promedio_NDVI.tif,EPSG:3115,2377,1169,30.0,-9999.0,60.6,0.1116,0.5114,0.2282,0.0723
4,SUSCEPTIBILIDAD,Susceptibilidad_promedio_EVI.tif,EPSG:3115,2377,1169,30.0,-9999.0,60.6,0.1179,0.5003,0.2375,0.0697
5,CHIRPS,CHIRPS_promedio_mensual_mm.tif,EPSG:3115,2377,1169,30.0,-9999.0,60.6,84.2151,157.9586,114.4907,21.3355



Ninguno de los rasters generados muestra cobertura anormalmente baja o vacia.


In [27]:

# --- Verificacion cruzada de resolucion/CRS entre factores ------------------
crs_unicos = tabla_validacion.crs.unique()
res_unicas = tabla_validacion.resolucion_m.unique()
print(f"CRS distintos entre los rasters generados: {list(crs_unicos)}")
print(f"Resoluciones distintas: {list(res_unicas)}")
assert (len(crs_unicos) == 1 and len(res_unicas) == 1), (
    "Los rasters generados NO comparten CRS/resolucion -- esto no deberia pasar (Seccion 6-8 trabajan en la grilla nativa comun)")
print("\nTodos los rasters (LS, A_USLE, Susceptibilidad, CHIRPS) comparten exactamente el mismo",
     "CRS/resolucion/dimensiones (grilla nativa 30m, EPSG:3115) -- se combinaron sin remuestreo,",
     "por lo que son directamente comparables pixel a pixel.")
if not tabla_val_puntual.empty:
    print("\nRecordatorio de la validacion de la Seccion 6: al combinar en la grilla nativa (sin",
         "remuestrear a una malla mas gruesa), la razon mediana raster/puntual quedo en",
         f"{tabla_val_puntual.razon_mediana.mean():.2f} (un primer intento con malla mas gruesa daba ~2.0).")
else:
    print("\n(La validacion puntual de la Seccion 6 se omitio -- CSV fuente no disponible; ver Seccion 6.)")


CRS distintos entre los rasters generados: ['EPSG:3115']
Resoluciones distintas: [np.float64(30.0)]

Todos los rasters (LS, A_USLE, Susceptibilidad, CHIRPS) comparten exactamente el mismo CRS/resolucion/dimensiones (grilla nativa 30m, EPSG:3115) -- se combinaron sin remuestreo, por lo que son directamente comparables pixel a pixel.

(La validacion puntual de la Seccion 6 se omitio -- CSV fuente no disponible; ver Seccion 6.)


## Sección 10 — Mapas y resultados organizados por categoría

Se generan mapas estáticos (PNG) de cada resultado principal, guardados en
la subcarpeta `MAPAS/` de su categoría correspondiente — nunca mezclados
entre categorías.


In [28]:

def mapa_rapido(ruta_tif, titulo, ruta_png, cmap="viridis"):
    with rasterio.open(ruta_tif) as ds:
        arr = ds.read(1).astype("float64")
        nod = ds.nodata
        arr = np.where(arr == nod, np.nan, arr) if nod is not None else arr
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(arr, cmap=cmap)
    ax.set_title(titulo, fontsize=11)
    ax.axis("off")
    plt.colorbar(im, ax=ax, shrink=0.7)
    plt.tight_layout()
    fig.savefig(ruta_png, dpi=150)
    plt.close(fig)
    return ruta_png

mapas_a_generar = [
    (ruta_ls_raster, "Factor LS (Mitasova, DEM real)", CATEGORIAS["LS"] / "MAPAS" / "Factor_LS.png", "YlOrRd"),
    (ruta_a_ndvi, "Perdida de suelo RUSLE - promedio (variante NDVI)", CATEGORIAS["RUSLE"] / "MAPAS" / "A_USLE_promedio_NDVI.png", "YlOrRd"),
    (ruta_susc_ndvi, "Susceptibilidad - promedio (variante NDVI)", CATEGORIAS["SUSC"] / "MAPAS" / "Susceptibilidad_promedio_NDVI.png", "YlOrRd"),
    (ruta_chirps, "Precipitacion CHIRPS - promedio mensual (mm)", CATEGORIAS["CHIRPS"] / "MAPAS" / "CHIRPS_promedio.png", "Blues"),
]
for ruta_tif, titulo, ruta_png, cmap in mapas_a_generar:
    ruta_png.parent.mkdir(parents=True, exist_ok=True)
    mapa_rapido(ruta_tif, titulo, ruta_png, cmap)
    categoria_reg = [c for c, p in [("FACTOR_LS", CATEGORIAS["LS"]), ("PERDIDA_SUELO_RUSLE", CATEGORIAS["RUSLE"]),
                                    ("SUSCEPTIBILIDAD", CATEGORIAS["SUSC"]), ("CHIRPS", CATEGORIAS["CHIRPS"])]
                    if str(p) in str(ruta_png)][0]
    registrar(categoria_reg, ruta_png, f"Mapa estatico: {titulo}")
    print(f"Guardado: {ruta_png}")


Guardado: D:\Diego Angrino Chiran\Documentos\Cenicaña\SOLIX\RUSLE_Amaime\RESULTADOS\01_FACTORES_RUSLE\FACTOR_LS\MAPAS\Factor_LS.png


Guardado: D:\Diego Angrino Chiran\Documentos\Cenicaña\SOLIX\RUSLE_Amaime\RESULTADOS\02_PERDIDA_SUELO_RUSLE\MAPAS\A_USLE_promedio_NDVI.png


Guardado: D:\Diego Angrino Chiran\Documentos\Cenicaña\SOLIX\RUSLE_Amaime\RESULTADOS\03_SUSCEPTIBILIDAD\MAPAS\Susceptibilidad_promedio_NDVI.png


Guardado: D:\Diego Angrino Chiran\Documentos\Cenicaña\SOLIX\RUSLE_Amaime\RESULTADOS\04_CHIRPS\MAPAS\CHIRPS_promedio.png


## Sección 11 — Resumen final del pipeline


In [29]:

tabla_archivos = pd.DataFrame(archivos_generados)
print(f"Total de archivos generados y registrados por este notebook: {len(tabla_archivos)}")
print(f"\nPor categoria:")
display(tabla_archivos.groupby("categoria").size().sort_values(ascending=False))

ruta_resumen = CATEGORIAS["VALIDACION"] / "INFORMES" / "Resumen_archivos_generados.csv"
tabla_archivos.to_csv(ruta_resumen, index=False, encoding="utf-8-sig")
print(f"\nIndice completo de archivos guardado en: {ruta_resumen}")


Total de archivos generados y registrados por este notebook: 29

Por categoria:


categoria
SUSCEPTIBILIDAD        5
CHIRPS                 4
PERDIDA_SUELO          4
FACTOR_LS              4
VALIDACION             3
BASE_DATOS             2
FACTOR_P               2
FACTOR_K               2
FACTOR_C               2
PERDIDA_SUELO_RUSLE    1
dtype: int64


Indice completo de archivos guardado en: D:\Diego Angrino Chiran\Documentos\Cenicaña\SOLIX\RUSLE_Amaime\RESULTADOS\06_VALIDACION\INFORMES\Resumen_archivos_generados.csv


In [30]:

resumen_texto = f"""
RESUMEN DE EJECUCION -- RUSLE_PIPELINE_COMPLETO.ipynb
Fecha de ejecucion: {FECHA_EJECUCION}

1) INSUMOS UTILIZADOS (ninguno modificado):
   - Excel original: {EXCEL_ORIGINAL.name} (117 parcelas x 24 columnas)
   - Excel tesis Dayana: {EXCEL_TESIS.name} (117 parcelas x 227 columnas, hoja 'Datos')
   - Metodologia: {DOCX_METODO.name}
   - AOI oficial: {AOI_GEOJSON.name} ({aoi_gdf.to_crs(CRS_PROYECTADO).area.iloc[0]/1e6:,.0f} km2)
   - Rasters reales integrados sin recalcular: LS_Mitasova_2001_CUENCA_COMPLETA.tif,
     Pendiente_grados_CUENCA_COMPLETA.tif, Flow_Accumulation_CUENCA_COMPLETA.tif,
     60 meses de CHIRPS y 60 meses de Sentinel-2 NDVI/EVI (mensual_cuenca_completa/)

2) FACTOR LS: metodologia Mitasova (DETALLE RESULTADOS.docx), clasificado
   segun Lu et al. (2020). Validado contra 'LS Diego' (tesis) en 117 puntos:
   correlacion {corr:.3f}, razon mediana {comp.razon.median():.2f} -- confirma
   que el raster reproduce fielmente la metodologia oficial.

3) HALLAZGO IMPORTANTE: los rasters individuales Factor_R/K/C/P.tif y
   A_USLE.tif del notebook ORIGINAL no existen en Salida/rasters/ (se
   busco exhaustivamente). K, C, P provienen de la tabla de 117 puntos de
   la tesis (K, P interpolados por IDW; C viene del NDVI/EVI mensual real).
   R y la perdida de suelo A se calcularon pixel a pixel con datos reales
   (CHIRPS, Sentinel-2) reutilizando la formula ya validada del geovisor,
   materializados aqui por primera vez como GeoTIFF independientes.
   UNIDADES: A_USLE_promedio_mensual_*.tif es el promedio de los 60 valores
   MENSUALES 2021-2025 (ton/ha/MES), no un total anual.

4) CHIRPS: diagnostico de cobertura dentro del AOI documentado en la
   Seccion 8 (ver Validacion_maestra_rasters.csv y CHIRPS_cobertura_por_mes.csv).
   Cobertura de datos dentro del AOI: {tabla_cobertura.pct_cobertura_dentro_aoi.min():.1f}%
   a {tabla_cobertura.pct_cobertura_dentro_aoi.max():.1f}% en los 60 meses -- el raster
   esta completo; los "espacios en blanco" reportados antes en el geovisor eran de
   renderizado (mascara por centro de celda), no de datos faltantes.

5) ARCHIVOS GENERADOS: {len(tabla_archivos)}, organizados en {len(CATEGORIAS)}
   categorias dentro de RESULTADOS/, sin mezclar factores/RUSLE/susceptibilidad/CHIRPS.

6) PROBLEMAS ENCONTRADOS Y CORREGIDOS EN ESTA REVISION:
   - Bug de merge (columnas LONG/LAT duplicadas al comparar LS Diego vs
     raster) -- corregido usando las columnas propias de la tesis.
   - Confirmada la NO existencia de rasters de factores individuales
     previos -- documentado explicitamente en vez de asumir o inventar.
   - Sesgo de ~2x en A_USLE detectado al remuestrear LS/CHIRPS/S2 a una
     malla mas gruesa (EPSG:4326, ~127m) antes de combinarlos: LS tiene
     mucha variabilidad local (picos en los cauces) y se distorsiona al
     remuestrear. CORREGIDO: se descubrio que LS, CHIRPS mensual y
     Sentinel-2 mensual ya comparten exactamente la misma grilla nativa
     (2377x1169 px, EPSG:3115, 30m) -- se combinan sin remuestrear.
     Razon mediana final raster/puntual validado: {"N/D (CSV de validacion puntual no disponible)" if tabla_val_puntual.empty else f"{tabla_val_puntual.razon_mediana.mean():.2f}"}
     (antes de la correccion: ~2.0).
"""
print(resumen_texto)
ruta_resumen_txt = CATEGORIAS["VALIDACION"] / "INFORMES" / "Resumen_final_pipeline.txt"
with open(ruta_resumen_txt, "w", encoding="utf-8") as f:
    f.write(resumen_texto)
registrar("VALIDACION", ruta_resumen_txt, "Resumen final del pipeline (texto)")
print(f"Guardado: {ruta_resumen_txt}")



RESUMEN DE EJECUCION -- RUSLE_PIPELINE_COMPLETO.ipynb
Fecha de ejecucion: 2026-09-15 08:32

1) INSUMOS UTILIZADOS (ninguno modificado):
   - Excel original: Cuenca amaime.xlsx (117 parcelas x 24 columnas)
   - Excel tesis Dayana: Cuenca Amaime Tesis Dayana.xlsx (117 parcelas x 227 columnas, hoja 'Datos')
   - Metodologia: DETALLE RESULTADOS.docx
   - AOI oficial: Cuenca_Amaime.geojson (1,510 km2)
   - Rasters reales integrados sin recalcular: LS_Mitasova_2001_CUENCA_COMPLETA.tif,
     Pendiente_grados_CUENCA_COMPLETA.tif, Flow_Accumulation_CUENCA_COMPLETA.tif,
     60 meses de CHIRPS y 60 meses de Sentinel-2 NDVI/EVI (mensual_cuenca_completa/)

2) FACTOR LS: metodologia Mitasova (DETALLE RESULTADOS.docx), clasificado
   segun Lu et al. (2020). Validado contra 'LS Diego' (tesis) en 117 puntos:
   correlacion 0.999, razon mediana 1.00 -- confirma
   que el raster reproduce fielmente la metodologia oficial.

3) HALLAZGO IMPORTANTE: los rasters individuales Factor_R/K/C/P.tif y
   A_USLE.